In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, subprocess

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = "reasoning anchor"
GOOGLE_DRIVE_PATH = os.path.join("/content/drive", "MyDrive", GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
assert os.path.isdir(GOOGLE_DRIVE_PATH), f"Not found: {GOOGLE_DRIVE_PATH}"

os.chdir(GOOGLE_DRIVE_PATH)
if GOOGLE_DRIVE_PATH not in sys.path:
    sys.path.append(GOOGLE_DRIVE_PATH)

uv_path = shutil.which("uv")
if uv_path is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-qU", "uv"], check=True)
uv_path = shutil.which("uv")
assert uv_path is not None, "uv install failed"

data_dir = os.path.join(GOOGLE_DRIVE_PATH, "data")
os.makedirs(data_dir, exist_ok=True)

Mounted at /content/drive


In [2]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
train_dataset = dataset["train"]
test_dataset = dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [4]:
!pip -q install json-repair
!pip -q install datasets sentence-transformers torch-geometric

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch_geometric.data import Data

from train import train_anchor_model, train
from model import GraphOfThoughtPrunerGraphSAGE

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
input_dimension = embedding_model.get_sentence_embedding_dimension()

def load_gsm8k_dataset(sample_size=50, split="train"):
    raw_dataset = load_dataset("openai/gsm8k", "main")[split]
    raw_dataset = raw_dataset.select(range(sample_size))
    dataset = []
    for item in raw_dataset:
        dataset.append({"question": item["question"], "answer": item["answer"]})
    return dataset

def build_data_function(graph, anchor_labels=None):
    node_id_to_index = {}
    node_features = []
    for index in range(len(graph["nodes"])):
        node = graph["nodes"][index]
        node_id_to_index[node["node_id"]] = index
        node_features.append(embedding_model.encode(node["text"]))
    edge_source_list = []
    edge_target_list = []
    for edge in graph["edges"]:
        if edge["source"] in node_id_to_index and edge["target"] in node_id_to_index:
            edge_source_list.append(node_id_to_index[edge["source"]])
            edge_target_list.append(node_id_to_index[edge["target"]])
    x = torch.tensor(node_features, dtype=torch.float32)
    edge_index = torch.tensor([edge_source_list, edge_target_list], dtype=torch.long)
    data = Data(x=x, edge_index=edge_index)
    if anchor_labels is not None:
        anchor_label = []
        for node in graph["nodes"]:
            anchor_label.append(anchor_labels.get(node["node_id"], 0.0))
        data.anchor_label = torch.tensor(anchor_label, dtype=torch.float32)
    return data

dataset = load_gsm8k_dataset(sample_size=10)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 5.1 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1600/1671001684.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  input_dimension = embedding_model.get_sentence_embedding_dimension()


In [5]:
!pip -q install transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

class HuggingFaceGenerationModel:
    def __init__(self, model_name="Qwen/Qwen3-4B-Instruct-2507", device=None, max_new_tokens=4096):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.device = device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if self.device == "cuda" else torch.float32, device_map="auto" if self.device == "cuda" else None, trust_remote_code=True)
        if self.device != "cuda":
            self.model = self.model.to(self.device)
        self.model.eval()

    def clean_response(self, response):
        response = response.strip()
        if "</think>" in response:
            response = response.split("</think>")[-1].strip()
        if response.startswith("```json"):
            response = response[len("```json"):].strip()
        if response.startswith("```"):
            response = response[len("```"):].strip()
        if response.endswith("```"):
            response = response[:-3].strip()
        return response

    def generate(self, prompt):
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False, pad_token_id=self.tokenizer.eos_token_id)
        generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        response = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return self.clean_response(response)

generation_model = HuggingFaceGenerationModel("Qwen/Qwen3-4B-Instruct-2507", max_new_tokens=4096)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class OldGraphOfThoughtPrunerGraphSAGE(nn.Module):
    def __init__(self, input_dimension, hidden_dimension=256, dropout_rate=0.2):
        super().__init__()

        self.input_layer = nn.Linear(input_dimension, hidden_dimension)

        self.first_graphsage_layer = SAGEConv(hidden_dimension, hidden_dimension)
        self.second_graphsage_layer = SAGEConv(hidden_dimension, hidden_dimension)

        self.dropout_layer = nn.Dropout(dropout_rate)

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dimension, hidden_dimension),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dimension, 1)
        )

    def forward(self, node_features, edge_index):
        hidden = F.relu(self.input_layer(node_features))
        hidden = self.dropout_layer(F.relu(self.first_graphsage_layer(hidden, edge_index)))
        hidden = self.dropout_layer(F.relu(self.second_graphsage_layer(hidden, edge_index)))
        return self.output_layer(hidden).squeeze(-1)

anchor_model = OldGraphOfThoughtPrunerGraphSAGE(input_dimension).to(device)
anchor_model.load_state_dict(torch.load("/content/drive/MyDrive/reasoning anchor/checkpoints/best_anchor_model_2.pt", map_location=device))
anchor_model.eval()

OldGraphOfThoughtPrunerGraphSAGE(
  (input_layer): Linear(in_features=384, out_features=256, bias=True)
  (first_graphsage_layer): SAGEConv(256, 256, aggr=mean)
  (second_graphsage_layer): SAGEConv(256, 256, aggr=mean)
  (dropout_layer): Dropout(p=0.2, inplace=False)
  (output_layer): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)

In [8]:
import json
import os
import time
import torch
import random
import numpy as np
import pandas as pd

os.chdir("/content/drive/MyDrive/reasoning anchor")

got_anchor_path = "/content/drive/MyDrive/reasoning anchor/data/gsm8k_got_anchor_qwen_long.jsonl"

got_anchor_dataset = []
with open(got_anchor_path, "r", encoding="utf-8") as file:
    for line in file:
        got_anchor_dataset.append(json.loads(line))

print("loaded got_anchor_dataset:", len(got_anchor_dataset))

from loss import training_loss, anchor_loss, structural_loss, deletion_loss
from train import prune_graph_by_top_ratio, get_anchors_from_anchor_model
from model import GraphOfThoughtPrunerGraphSAGE

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def to_float(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu().item())
    return float(value)

eval_dataset = got_anchor_dataset
keep_ratios = [0.5, 0.7, 0.9]
anchor_threshold = 0.5

random_repeat_count = 5
random_repeat_seeds = [1234, 2024, 42, 7, 999]

checkpoint_specs = {
    "gnn_3_layer_1e3": {
        "block_count": 3,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial.pt"
        ]
    },
    "gnn_2_layer_1e3": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2.pt"
        ]
    },
    "gnn_5_layer_1e4": {
        "block_count": 5,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_3_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_2.pt"
        ]
    },
    "gnn_4_layer_1e3": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_4_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_4_new_2.pt"
        ]
    },
    "gnn_4_layer_1e4": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_5_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_5_new_2.pt"
        ]
    },
    "gnn_4_layer_1e5": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_6_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_6_new_2.pt"
        ]
    },
    "gnn_2_layer_1e4": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_2_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2_new_2.pt"
        ]
    },
    "gnn_5_layer_1e4_new_gated_1": {
        "block_count": 5,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_1.pt"
        ]
    },
    "gnn_4_layer_1e3_new_gated_2": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_2.pt"
        ]
    },
    "gnn_4_layer_1e4_new_gated_3": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_3.pt"
        ]
    },
    "gnn_4_layer_1e5_new_gated_4": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_4.pt"
        ]
    },
    "gnn_2_layer_1e4_new_gated_5": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_5.pt"
        ]
    }
}

seed_everything(1234)

gnn_models = {}

for model_name, spec in checkpoint_specs.items():
    checkpoint_path = None

    for path in spec["paths"]:
        if os.path.exists(path):
            checkpoint_path = path
            break

    if checkpoint_path is None:
        print("skip missing checkpoint:", model_name)
        continue

    loaded_model = GraphOfThoughtPrunerGraphSAGE(
        input_dimension,
        block_count=spec["block_count"]
    ).to(device)

    loaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    loaded_model.eval()

    gnn_models[model_name] = loaded_model
    print("loaded:", model_name, checkpoint_path)

deterministic_methods = ["no_pruning"] + list(gnn_models.keys())
random_methods = ["random_pruning"]

def get_anchors_for_graph(line):
    graph = line["graph"]
    anchors = []

    if "anchor_labels" in line:
        for node in graph["nodes"]:
            if line["anchor_labels"].get(node["node_id"], 0.0) >= anchor_threshold:
                anchors.append(node)
    else:
        data = build_data_function(graph)
        if device is not None:
            data = data.to(device)

        anchors = get_anchors_from_anchor_model(
            anchor_model,
            graph,
            data,
            anchor_threshold
        )

    return anchors

anchor_cache = {}

for index, line in enumerate(eval_dataset):
    anchor_cache[index] = get_anchors_for_graph(line)

print("cached anchors:", len(anchor_cache))

def build_random_pruned_graph(graph, keep_ratio):
    keep_count = max(1, int(len(graph["nodes"]) * keep_ratio))
    keep_indices = random.sample(range(len(graph["nodes"])), keep_count)

    keep_node_ids = []
    for index in keep_indices:
        keep_node_ids.append(graph["nodes"][index]["node_id"])

    keep_node_id_set = set(keep_node_ids)

    pruned_graph = {
        "nodes": [],
        "edges": []
    }

    for node in graph["nodes"]:
        if node["node_id"] in keep_node_id_set:
            pruned_graph["nodes"].append(node)

    for edge in graph["edges"]:
        if edge["source"] in keep_node_id_set and edge["target"] in keep_node_id_set:
            pruned_graph["edges"].append(edge)

    return pruned_graph

def build_gnn_pruned_graph(method, graph, keep_ratio):
    data = build_data_function(graph)
    if device is not None:
        data = data.to(device)

    current_model = gnn_models[method]
    current_model.eval()

    with torch.no_grad():
        node_keep_score = current_model(data.x, data.edge_index)
        node_keep_probability = torch.sigmoid(node_keep_score)

    pruned_graph = prune_graph_by_top_ratio(
        graph,
        node_keep_probability,
        keep_ratio
    )

    return pruned_graph

def evaluate_one_method_once(method, repeat_index, seed):
    seed_everything(seed)

    rows = []
    start_time = time.perf_counter()

    for line_index, line in enumerate(eval_dataset):
        question = line["question"]
        graph = line["graph"]
        anchors = anchor_cache[line_index]

        for keep_ratio in keep_ratios:
            item_start_time = time.perf_counter()

            if method == "no_pruning":
                pruned_graph = graph

            elif method == "random_pruning":
                pruned_graph = build_random_pruned_graph(graph, keep_ratio)

            else:
                pruned_graph = build_gnn_pruned_graph(method, graph, keep_ratio)

            a_loss = anchor_loss(
                pruned_graph,
                graph,
                question,
                generation_model,
                embedding_model
            )

            s_loss = structural_loss(
                pruned_graph,
                graph,
                anchors
            )

            d_loss = deletion_loss(
                pruned_graph,
                graph
            )

            score = training_loss(
                pruned_graph,
                graph,
                question,
                generation_model,
                embedding_model,
                anchors
            )

            item_elapsed_time = time.perf_counter() - item_start_time

            node_ratio = len(pruned_graph["nodes"]) / len(graph["nodes"]) if len(graph["nodes"]) > 0 else 0.0
            edge_ratio = len(pruned_graph["edges"]) / len(graph["edges"]) if len(graph["edges"]) > 0 else 0.0

            rows.append({
                "repeat": repeat_index,
                "seed": seed,
                "method": method,
                "line_index": line_index,
                "keep_ratio": keep_ratio,
                "evaluation_score": to_float(score),
                "anchor_loss": to_float(a_loss),
                "structural_loss": to_float(s_loss),
                "deletion_loss": to_float(d_loss),
                "kept_node_ratio": float(node_ratio),
                "kept_edge_ratio": float(edge_ratio),
                "item_elapsed_time_seconds": float(item_elapsed_time)
            })

    elapsed_time = time.perf_counter() - start_time
    print("finished:", method, "repeat:", repeat_index, "seed:", seed, "time:", elapsed_time)

    return rows

raw_rows = []
total_start_time = time.perf_counter()

for method in deterministic_methods:
    rows = evaluate_one_method_once(
        method=method,
        repeat_index=1,
        seed=1234
    )
    raw_rows.extend(rows)

for repeat_index in range(random_repeat_count):
    seed = random_repeat_seeds[repeat_index % len(random_repeat_seeds)]
    rows = evaluate_one_method_once(
        method="random_pruning",
        repeat_index=repeat_index + 1,
        seed=seed
    )
    raw_rows.extend(rows)

total_elapsed_time = time.perf_counter() - total_start_time

raw_df = pd.DataFrame(raw_rows)

metrics = [
    "evaluation_score",
    "anchor_loss",
    "structural_loss",
    "deletion_loss",
    "kept_node_ratio",
    "kept_edge_ratio",
    "item_elapsed_time_seconds"
]

summary_rows = []

for method in raw_df["method"].unique():
    method_df = raw_df[raw_df["method"] == method]

    row = {
        "method": method,
        "row_count": len(method_df),
        "repeat_count": method_df["repeat"].nunique(),
        "eval_dataset_size": len(eval_dataset),
        "keep_ratio_count": len(keep_ratios),
        "total_time_seconds": float(method_df["item_elapsed_time_seconds"].sum()),
        "total_time_minutes": float(method_df["item_elapsed_time_seconds"].sum() / 60.0)
    }

    for metric in metrics:
        values = method_df[metric].astype(float).values

        row[metric + "_mean"] = float(np.mean(values))
        row[metric + "_std"] = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
        row[metric + "_min"] = float(np.min(values))
        row[metric + "_max"] = float(np.max(values))
        row[metric + "_median"] = float(np.median(values))

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("evaluation_score_mean").reset_index(drop=True)

save_raw_path = "/content/drive/MyDrive/reasoning anchor/data/pruning_model_fast_repeated_eval_raw.csv"
save_summary_path = "/content/drive/MyDrive/reasoning anchor/data/pruning_model_fast_repeated_eval_summary.csv"
save_json_path = "/content/drive/MyDrive/reasoning anchor/data/pruning_model_fast_repeated_eval_results.json"

raw_df.to_csv(save_raw_path, index=False)
summary_df.to_csv(save_summary_path, index=False)

with open(save_json_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            "random_repeat_count": random_repeat_count,
            "random_repeat_seeds": random_repeat_seeds,
            "deterministic_methods": deterministic_methods,
            "eval_dataset_size": len(eval_dataset),
            "keep_ratios": keep_ratios,
            "anchor_threshold": anchor_threshold,
            "total_elapsed_time_seconds": total_elapsed_time,
            "total_elapsed_time_minutes": total_elapsed_time / 60.0,
            "raw_results": raw_df.to_dict(orient="records"),
            "summary_results": summary_df.to_dict(orient="records")
        },
        file,
        ensure_ascii=False,
        indent=2
    )

print("saved raw results:", save_raw_path)
print("saved summary results:", save_summary_path)
print("saved json results:", save_json_path)
print("total elapsed time seconds:", total_elapsed_time)
print("total elapsed time minutes:", total_elapsed_time / 60.0)

summary_df

loaded got_anchor_dataset: 100
loaded: gnn_3_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial.pt
loaded: gnn_2_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2.pt
loaded: gnn_5_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_2.pt
loaded: gnn_4_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_4_new_2.pt
loaded: gnn_4_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_5_new_2.pt
loaded: gnn_4_layer_1e5 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_6_new_2.pt
loaded: gnn_2_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2_new_2.pt
loaded: gnn_5_layer_1e4_new_gated_1 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_1.pt
loaded: gnn_4_layer_1e3_new_gated_2 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_2.pt
loaded: gnn_4_layer_1e4_new_g

,method,row_count,repeat_count,eval_dataset_size,keep_ratio_count,total_time_seconds,total_time_minutes,evaluation_score_mean,evaluation_score_std,evaluation_score_min,...,kept_edge_ratio_mean,kept_edge_ratio_std,kept_edge_ratio_min,kept_edge_ratio_max,kept_edge_ratio_median,item_elapsed_time_seconds_mean,item_elapsed_time_seconds_std,item_elapsed_time_seconds_min,item_elapsed_time_seconds_max,item_elapsed_time_seconds_median
0,gnn_4_layer_1e4,300,1,100,3,2963.421418,49.390357,1.182885,0.156212,0.765607,...,0.805609,0.217155,0.000000,1.0,0.857143,9.878071,2.052286,1.331013,18.095143,9.968051
1,gnn_4_layer_1e5_new_gated_4,300,1,100,3,3044.958768,50.749313,1.185714,0.130658,0.833027,...,0.874049,0.173372,0.250000,1.0,1.000000,10.149863,1.261103,1.730611,13.619508,9.978634
2,gnn_4_layer_1e3_new_gated_2,300,1,100,3,2898.508960,48.308483,1.188566,0.184271,0.700724,...,0.770248,0.286563,0.000000,1.0,0.888889,9.661697,2.383337,1.265657,14.525272,9.870657
3,gnn_4_layer_1e4_new_gated_3,300,1,100,3,3065.727591,51.095460,1.189642,0.125358,0.720526,...,0.908228,0.133968,0.454545,1.0,1.000000,10.219092,1.349847,1.765073,16.713131,9.988982
4,gnn_4_layer_1e5,300,1,100,3,3043.271471,50.721191,1.194049,0.115118,0.905719,...,0.893094,0.155858,0.333333,1.0,1.000000,10.144238,1.649075,1.305193,13.589407,10.015164
5,gnn_2_layer_1e4,300,1,100,3,3042.611050,50.710184,1.194860,0.127520,0.813586,...,0.888521,0.151432,0.333333,1.0,1.000000,10.142037,1.575890,1.705369,18.140221,9.962634
6,gnn_4_layer_1e3,300,1,100,3,2881.142140,48.019036,1.195498,0.192248,0.681072,...,0.732516,0.293515,0.000000,1.0,0.806250,9.603807,2.707122,1.370068,16.879918,9.990036
7,gnn_2_layer_1e4_new_gated_5,300,1,100,3,3056.521729,50.942029,1.196356,0.121383,0.850669,...,0.912934,0.131750,0.421053,1.0,1.000000,10.188406,1.436329,1.735247,18.076177,10.007854
8,gnn_5_layer_1e4,300,1,100,3,3042.159832,50.702664,1.196825,0.114444,0.846769,...,0.900739,0.147923,0.428571,1.0,1.000000,10.140533,1.316107,1.702986,15.341433,9.923490
9,gnn_5_layer_1e4_new_gated_1,300,1,100,3,3047.055987,50.784266,1.197036,0.119869,0.803834,...,0.886317,0.160702,0.259259,1.0,1.000000,10.156853,1.661351,1.692977,17.738712,10.064277
